<a href="https://colab.research.google.com/github/Ravindra1972/Anaytics-in-finance-using-Python/blob/main/Bond_Yield_Strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import pandas_datareader.data as web


def download_etf_prices(start="2005-01-01"):

    tickers = ["TLT", "SHY"]

    prices = yf.download(
        tickers,
        start=start,
        progress=False
    )["Close"]

    prices = prices.dropna()

    return prices


def download_yields(start="2005-01-01"):

    fred_series = ["DGS10", "DGS2"]

    yields = web.DataReader(
        fred_series,
        "fred",
        start
    )

    yields = yields.dropna()

    return yields



In [1]:
import numpy as np


def calculate_metrics(series):

    returns = series.pct_change(fill_method=None).dropna()

    cagr = (series.iloc[-1] ** (252 / len(series))) - 1

    sharpe = returns.mean() / returns.std() * np.sqrt(252)

    drawdown = series / series.cummax() - 1

    max_dd = drawdown.min()

    print("CAGR:", round(cagr * 100, 2), "%")
    print("Sharpe:", round(sharpe, 2))
    print("Max Drawdown:", round(max_dd * 100, 2), "%")



In [2]:
import matplotlib.pyplot as plt


def plot_results(strategy, benchmark):

    plt.figure(figsize=(10,6))

    plt.plot(strategy, label="Yield Curve Strategy")
    plt.plot(benchmark, label="TLT Benchmark")

    plt.legend()

    plt.title("Bond Strategy vs Benchmark")

    plt.savefig("results/performance.png")

    plt.show()



In [3]:
def compute_yield_curve_signal(yields):

    slope = yields["DGS10"] - yields["DGS2"]

    signal = slope > 0

    return signal


In [4]:
import pandas as pd


def run_backtest(prices, signal):

    returns = prices.pct_change(fill_method=None).dropna()

    strategy_returns = []

    for date in returns.index:

        if date not in signal.index:
            continue

        if signal.loc[date]:
            strategy_returns.append(returns.loc[date, "TLT"])
        else:
            strategy_returns.append(returns.loc[date, "SHY"])

    strategy_returns = pd.Series(strategy_returns, index=returns.index[:len(strategy_returns)])

    cumulative = (1 + strategy_returns).cumprod()

    return cumulative

